In [0]:
CREATE OR REPLACE VIEW supermercadoetl_prod.gold.costos_recetas AS
SELECT 
    r.receta,
    r.ingrediente,
    c.nombre AS nombre_producto_completo,
    c.descripcion,
    c.categoria,
    c.precio_normalizado AS precio_producto,
    c.cantidad_base,
    c.precio_unitario,
    c.unidad,
    r.cantidad_necesaria,
    r.unidad_receta,
    -- Cálculo del costo del ingrediente
    ROUND(c.precio_unitario * r.cantidad_necesaria, 2) AS costo_ingrediente,
    r.rendimiento_porciones,
    c.fecha_extraccion,
    -- Score de match para priorizar el mejor
    CASE 
      WHEN TRIM(LOWER(c.nombre)) = TRIM(LOWER(r.ingrediente)) THEN 1  -- Match exacto
      WHEN TRIM(LOWER(c.nombre)) LIKE CONCAT(TRIM(LOWER(r.ingrediente)), '%') THEN 2  -- Empieza con
      ELSE 3  -- Contiene
    END AS match_score,
    LENGTH(c.nombre) as nombre_length,
    c.url
  FROM supermercadoetl_prod.gold.catalogo_productos c
  INNER JOIN supermercadoetl_prod.gold.recetas r
    ON TRIM(LOWER(c.nombre)) LIKE CONCAT('%', TRIM(LOWER(r.ingrediente)), '%')
  -- Priorizar: (1) mejor match_score, (2) nombre más corto (más específico)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.receta, r.ingrediente, c.fecha_extraccion 
    ORDER BY match_score ASC, nombre_length ASC
  ) = 1
